# Netflix Personalized Content Discovery Engine
### Google Colab GPU-Accelerated Training & Evaluation Pipeline

This notebook trains **5 recommendation models** on the real **Netflix Prize Dataset** (100M+ ratings):

| # | Model | Type | Description |
|---|-------|------|-------------|
| 1 | **Funk SVD** | Matrix Factorization | Learns latent user/item factors via SGD |
| 2 | **Item-Based CF** | Collaborative Filtering | Cosine similarity between item rating patterns |
| 3 | **User-Based CF** | Collaborative Filtering | Finds users with matching watch history, cross-recommends |
| 4 | **Neural CF (NeuMF)** | Deep Learning | GMF + MLP branches on GPU via PyTorch |
| 5 | **Content-Based** | TF-IDF Similarity | Title-word + release era similarity matching |

Go to **Runtime → Change runtime type** and select **T4 GPU** (or higher).

---
## Step 1: Upload Project Files

Upload the project zip file (`recommendation_system.zip`) to Colab and extract it.

In [1]:
# Option A: Upload the zip file manually
from google.colab import files
import os

# Upload the zip file
print("Please upload 'recommendation_system.zip':")
uploaded = files.upload()

# Extract
!unzip -qo recommendation_system.zip -d /content/recommendation_system
os.chdir('/content/recommendation_system')
print("\n\u2705 Project extracted! Working directory:", os.getcwd())
!ls -la

Please upload 'recommendation_system.zip':


Saving recommendation_system.zip to recommendation_system.zip

✅ Project extracted! Working directory: /content/recommendation_system
total 12
drwxr-xr-x 3 root root 4096 Jun 12 16:12  .
drwxr-xr-x 1 root root 4096 Jun 12 16:12  ..
drwxrwxrwx 5 root root 4096 Jun 12 15:18 'recommendation system'


---
## Step 2: Install Dependencies

Install all required Python libraries. PyTorch is pre-installed in Colab.

In [2]:
!pip install -q fpdf2 pandas numpy scipy matplotlib requests tqdm scikit-learn flask

# Verify GPU availability
import torch
print(f"\n\u2705 PyTorch version: {torch.__version__}")
print(f"\u2705 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"\u2705 GPU: {torch.cuda.get_device_name(0)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 12.7 MB/s eta 0:00:00

✅ PyTorch version: 2.11.0+cu128
✅ CUDA available: True
✅ GPU: Tesla T4


In [3]:
# Download and extract the Netflix Prize Dataset
!pip install -q kaggle
!kaggle datasets download -d netflix-inc/netflix-prize-data

# Create data directories and extract
!mkdir -p data/raw data/processed
!unzip -qo netflix-prize-data.zip -d data/raw
!rm -f netflix-prize-data.zip

print("\n\u2705 Netflix Prize Dataset downloaded and extracted!")
print("\nFiles in data/raw/:")
!ls -lh data/raw/

Dataset URL: https://www.kaggle.com/datasets/netflix-inc/netflix-prize-data
License(s): other
100% 683M/683M [00:07<00:00, 90.3MB/s]


✅ Netflix Prize Dataset downloaded and extracted!

Files in data/raw/:
total 2.0G
-rw-r--r-- 1 root root 473M Nov 13  2019 combined_data_1.txt
-rw-r--r-- 1 root root 530M Nov 13  2019 combined_data_2.txt
-rw-r--r-- 1 root root 444M Nov 13  2019 combined_data_3.txt
-rw-r--r-- 1 root root 527M Nov 13  2019 combined_data_4.txt
-rw-r--r-- 1 root root 565K Nov 13  2019 movie_titles.csv
-rw-r--r-- 1 root root  11M Nov 13  2019 probe.txt
-rw-r--r-- 1 root root  51M Nov 13  2019 qualifying.txt
-rw-r--r-- 1 root root 5.8K Nov 13  2019 README


In [7]:
!mv /content/recommendation_system/recommendation_system/* \
    /content/recommendation_system/

mv: cannot move '/content/recommendation_system/recommendation_system/data' to '/content/recommendation_system/data': Directory not empty


---
## Step 4: Parse Netflix Data & Build Content Features

This step:
1. **Parses** `combined_data_1.txt` (24M ratings from the first file)
2. **Maps** user/movie IDs to contiguous indices
3. **Splits** chronologically: 80% train / 20% test per user
4. **Builds** content features from movie titles (TF-IDF on title words + release decade)

**💡 Tip:** To load more ratings, set `NETFLIX_RATING_LIMIT` (default: 2,000,000). Set to `0` for all ~100M ratings.

In [8]:
import os
os.chdir('/content/recommendation_system')

# Set rating limit (default 2M for fast iteration; set to 0 for full dataset)
os.environ['NETFLIX_RATING_LIMIT'] = '2000000'

!python src/data_pipeline.py

Parsing Netflix Prize ratings files...
Rating limit set to 2,000,000 (set NETFLIX_RATING_LIMIT=0 for unlimited)
Reading combined_data_1.txt...
Reached subset limit of 2,000,000 ratings. Stopping parsing.
Total ratings loaded: 2,000,000
Mapping IDs to contiguous indices...
Splitting data chronologically per user...
Train ratings: 1,626,828 (81.3%)
Test ratings: 373,172 (18.7%)
Sparsity: 98.3822%
Processed datasets and index mappings saved to data/processed/.
Building content features from movie titles...
Content features built for 361 movies. Saved to movie_content_features.csv


In [14]:
# Verify the processed data
import json
with open('data/processed/metadata.json') as f:
    meta = json.load(f)

print("\n ca Dataset Statistics:")
print(f"   Users:    {int(meta['num_users']):,}")
print(f"   Movies:   {int(meta['num_movies']):,}")
print(f"   Ratings:  {int(meta['num_ratings']):,}")
print(f"   Sparsity: {meta['sparsity']*100:.2f}%")
print(f"   Train:    {int(meta['train_size']):,}")
print(f"   Test:     {int(meta['test_size']):,}")

# Check content features
# import pandas as pd
# content_df = pd.read_csv('data/processed/movie_content_features.csv')
# print(f"\n features built for {len(content_df)} movies")
# print("\n content strings:")
# print(content_df[['title', 'content_string']].head(10).to_string(index=False))


 ca Dataset Statistics:
   Users:    342,445
   Movies:   361
   Ratings:  2,000,000
   Sparsity: 98.38%
   Train:    1,626,828
   Test:     373,172


---
## Step 5: Train GPU Model (PyTorch NeuMF)

Trains the **Neural Matrix Factorization (NeuMF)** model on the GPU.  
This combines GMF (element-wise product) and MLP (deep layers) branches.

In [15]:
os.chdir('/content/recommendation_system')
!python src/pytorch_gpu_model.py

Using device: cuda
--------------------------------------------------
Training PyTorch Funk SVD model on GPU...
--------------------------------------------------
Epoch 1/15 - Train RMSE: 1.0321 | Test RMSE: 1.0204
Epoch 2/15 - Train RMSE: 1.0315 | Test RMSE: 1.0194
Epoch 3/15 - Train RMSE: 1.0316 | Test RMSE: 1.0194
Epoch 4/15 - Train RMSE: 1.0316 | Test RMSE: 1.0198
Epoch 5/15 - Train RMSE: 1.0315 | Test RMSE: 1.0201
Epoch 6/15 - Train RMSE: 1.0315 | Test RMSE: 1.0201
Epoch 7/15 - Train RMSE: 1.0315 | Test RMSE: 1.0198
Epoch 8/15 - Train RMSE: 1.0315 | Test RMSE: 1.0189
Epoch 9/15 - Train RMSE: 1.0316 | Test RMSE: 1.0197
Epoch 10/15 - Train RMSE: 1.0315 | Test RMSE: 1.0204
Epoch 11/15 - Train RMSE: 1.0315 | Test RMSE: 1.0206
Epoch 12/15 - Train RMSE: 1.0316 | Test RMSE: 1.0192
Epoch 13/15 - Train RMSE: 1.0315 | Test RMSE: 1.0208
Epoch 14/15 - Train RMSE: 1.0316 | Test RMSE: 1.0204
Epoch 15/15 - Train RMSE: 1.0315 | Test RMSE: 1.0205
PyTorch SVD trained in 387.91 seconds!

-----------

---
## Step 6: Train & Evaluate All 5 Models

This runs the full evaluation pipeline:
1. **Funk SVD** — Matrix factorization with biases (15 epochs SGD)
2. **Item-CF** — Item-item cosine similarity with shrinkage
3. **User-CF** — User-user centered cosine similarity
4. **Neural-CF** — Loads pre-trained NeuMF weights from Step 5
5. **Content-Based** — TF-IDF on movie titles + decade features

Metrics computed: **RMSE**, **MAE**, **MAP@10**

⚠️ This step takes the longest (~20-40 min depending on dataset size). The MAP@10 computation evaluates every test user.

In [ ]:
os.chdir('/content/recommendation_system')
!python -m src.evaluate

Starting PyTorch models training on GPU...
Using device: cuda
--------------------------------------------------
Training PyTorch Funk SVD model on GPU...
--------------------------------------------------
Epoch 1/15 - Train RMSE: 1.0321 | Test RMSE: 1.0200
Epoch 2/15 - Train RMSE: 1.0315 | Test RMSE: 1.0185
Epoch 3/15 - Train RMSE: 1.0316 | Test RMSE: 1.0195
Epoch 4/15 - Train RMSE: 1.0315 | Test RMSE: 1.0194
Epoch 5/15 - Train RMSE: 1.0315 | Test RMSE: 1.0200
Epoch 6/15 - Train RMSE: 1.0315 | Test RMSE: 1.0209
Epoch 7/15 - Train RMSE: 1.0315 | Test RMSE: 1.0195


In [ ]:
# Display results in a nice table
import json
import pandas as pd

with open('data/processed/results.json') as f:
    results = json.load(f)

results_df = pd.DataFrame(results).T
results_df.index.name = 'Model'
print("\n Model Performance Comparison:")
print("=" * 50)
print(results_df.to_string())
print("=" * 50)

# Highlight best model per metric
print(f"\n\ud83e\udd47 Best RMSE:   {results_df['RMSE'].idxmin()} ({results_df['RMSE'].min():.4f})")
print(f"\ud83e\udd47 Best MAE:    {results_df['MAE'].idxmin()} ({results_df['MAE'].min():.4f})")
print(f"\ud83e\udd47 Best MAP@10: {results_df['MAP@10'].idxmax()} ({results_df['MAP@10'].max():.4f})")

---
## Step 7: Launch Interactive Web Dashboard

Start the Flask server and open the AURA-REC dashboard to:
- 👤 Inspect user rating histories
- 🎬 Compare recommendations from all 5 models side-by-side
- 🌱 Test cold-start scenarios for new users
- 🔍 Search for similar movies

In [ ]:
import subprocess
import time
from google.colab.output import serve_kernel_port

os.chdir('/content/recommendation_system')

# Start Flask backend server in background
flask_process = subprocess.Popen(["python", "-m", "src.app"])
time.sleep(4)  # Let server initialize

print("\n\u2705 Flask dashboard is running!")
print("\ud83d\udc47 Click the link below to open the AURA-REC interface:\n")
serve_kernel_port(5000)

---
## Step 9: Run Unit Tests (Optional)

Validate that all models and the evaluation pipeline work correctly.

In [ ]:
os.chdir('/content/recommendation_system')
!python -m unittest discover tests/ -v